**Features Extraction**

In [2]:
import os
from io import BytesIO
from PIL import Image
import librosa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import cv2
import warnings
matplotlib.use('Agg')

data_path_Train = pd.read_csv("CSVs\\ravdess_train.csv")
data_path_Val = pd.read_csv("CSVs\\ravdess_val.csv")
data_path_Test = pd.read_csv("CSVs\\ravdess_test.csv")

In [18]:
def extract_features(ef_data: np.ndarray, ef_sample_rate: int) -> np.ndarray:

    ef_features = np.array([])
    n_fft = min(2048, len(ef_data))
    n_mels = min(128, n_fft // 2)

    # ZCR
    zcr = np.mean(librosa.feature.zero_crossing_rate(ef_data).T, axis=0)
    ef_features = np.hstack((ef_features, zcr))

    # Chroma STFT
    stft = librosa.stft(ef_data, n_fft=n_fft)
    chroma_stft = np.mean(
        librosa.feature.chroma_stft(S=np.abs(stft), sr=ef_sample_rate, n_fft=n_fft).T, axis=0
    )
    ef_features = np.hstack((ef_features, chroma_stft))

    # MFCC
    mfcc = np.mean(librosa.feature.mfcc(y=ef_data, sr=ef_sample_rate, n_fft=n_fft).T, axis=0)
    ef_features = np.hstack((ef_features, mfcc))

    # Root Mean Square Value
    rmse = np.mean(librosa.feature.rms(y=ef_data).T, axis=0)
    ef_features = np.hstack((ef_features, rmse))

    # Mel Spectrogram
    mel = np.mean(
        librosa.feature.melspectrogram(y=ef_data, sr=ef_sample_rate, n_fft=n_fft, n_mels=n_mels).T, axis=0
    )
    ef_features = np.hstack((ef_features, mel))

    return ef_features


def get_features(gf_path: str) -> np.ndarray:
    data, sample_rate = librosa.load(gf_path, sr=None)

    sample_rate = int(sample_rate)
    features = extract_features(data, sample_rate)

    return features


MIN_SAMPLES = 2048

def prepare_audios(pa_df: pd.DataFrame, pa_name: str):
    data = []
    labels = []
    skipped = 0
    total = len(pa_df)

    for _, row in pa_df.iterrows():
        try:
            audio, sr = librosa.load(row["path"], sr=None)

            if len(audio) < MIN_SAMPLES:
                skipped += 1
                print(f"Skipped (too short): {row['path']} ({len(audio)} samples)")
                continue

            features = extract_features(audio, int(sr))
            data.append(features)
            labels.append(row["emotion"])
            print(f"{pa_name} - Done! Saved {len(data)}/{total}", end="\r", flush=True)

        except Exception as e:
            print(f"Error processing {row['path']}: {e}")

    print(f"\n{pa_name} - Hoàn tất: {len(data)} files, bỏ qua: {skipped} files")

    data = np.array(data)
    labels = np.array(labels)

    os.makedirs("features", exist_ok=True)
    np.save(f"features\\{pa_name}_data.npy", data)
    np.save(f"features\\{pa_name}_labels.npy", labels)


prepare_audios(data_path_Train, "train")
prepare_audios(data_path_Val, "val")
prepare_audios(data_path_Test, "test")

Skipped (too short): Ravdess_Augmented\2004.wav (1257 samples)
Skipped (too short): Ravdess_Augmented\2005.wav (1257 samples)
Skipped (too short): Ravdess_Augmented\2006.wav (1571 samples)
Skipped (too short): Ravdess_Augmented\4584.wav (155 samples)
Skipped (too short): Ravdess_Augmented\1952.wav (596 samples)
Skipped (too short): Ravdess_Augmented\4586.wav (194 samples)
Skipped (too short): Ravdess_Augmented\1955.wav (596 samples)
Skipped (too short): Ravdess_Augmented\4585.wav (155 samples)
Skipped (too short): Ravdess_Augmented\1954.wav (745 samples)
Skipped (too short): Ravdess_Augmented\4651.wav (221 samples)
Skipped (too short): Ravdess_Augmented\4649.wav (221 samples)
train - Done! Saved 4136/4147
train - Hoàn tất: 4136 files, bỏ qua: 11 files
Skipped (too short): Ravdess_Augmented\4648.wav (221 samples)
Skipped (too short): Ravdess_Augmented\4587.wav (155 samples)
Skipped (too short): Ravdess_Augmented\2007.wav (1257 samples)
val - Done! Saved 1034/1037
val - Hoàn tất: 1034 fi

**Create spectrogram image**

In [4]:
location = "features\\images\\"
MIN_SAMPLES = 2048


def graph_spectrogram(gs_audio, sr: int | float | None = None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        if isinstance(gs_audio, str):
            audio, loaded_sr = librosa.load(gs_audio, sr=None)
            sr = int(loaded_sr)
        else:
            audio = gs_audio
            if sr is None:
                raise ValueError(
                    "sr must be provided when gs_audio is a waveform array."
                )
            sr = int(sr)

        mel = librosa.feature.melspectrogram(y=audio, sr=sr)
        mel_db = librosa.power_to_db(mel, ref=np.max)

        fig, ax = plt.subplots(1)
        fig.set_size_inches(2.24, 2.24)
        fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
        ax.axis("off")

        librosa.display.specshow(mel_db, ax=ax, cmap="inferno")

        fig.canvas.draw()
        buf = BytesIO()
        fig.savefig(buf, format="png")
        buf.seek(0)
        img = np.array(Image.open(buf))[:, :, :3]
        plt.close(fig)

        return img


def prepare_images(pi_df: pd.DataFrame, pi_Name: str):
    save_dir = os.path.join(location, pi_Name)
    os.makedirs(save_dir, exist_ok=True)

    file_data = []
    skipped = 0
    counter = 1

    for _, audio in pi_df.iterrows():
        try:
            raw, sr = librosa.load(audio["path"], sr=None)

            if len(raw) < MIN_SAMPLES:
                skipped += 1
                print(f"\nSkipped (too short): {audio['path']} ({len(raw)} samples)")
                continue

            img = graph_spectrogram(raw, sr=int(sr))
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            img = cv2.resize(img, (224, 224))

            img_path = os.path.join(save_dir, f"{counter}.png")
            cv2.imwrite(img_path, img)

            file_data.append([img_path, audio["emotion"]])
            print(f"\r{pi_Name} - Saved {counter}/{len(pi_df)}", end="", flush=True)
            counter += 1

        except Exception as e:
            print(f"\nError: {audio['path']} → {e}")

    print(f"\n{pi_Name} Done — {counter - 1} samples, bỏ qua: {skipped} files")

    result_df = pd.DataFrame(file_data, columns=["path", "emotion"])
    result_df.to_csv(f"CSVs\\{pi_Name}_images.csv", index=False)


prepare_images(data_path_Train, "train")
prepare_images(data_path_Val, "val")
prepare_images(data_path_Test, "test")

train - Saved 2065/4147
Skipped (too short): Ravdess_Augmented\2004.wav (1257 samples)
train - Saved 2532/4147
Skipped (too short): Ravdess_Augmented\2005.wav (1257 samples)
train - Saved 2567/4147
Skipped (too short): Ravdess_Augmented\2006.wav (1571 samples)
train - Saved 2587/4147
Skipped (too short): Ravdess_Augmented\4584.wav (155 samples)
train - Saved 3084/4147
Skipped (too short): Ravdess_Augmented\1952.wav (596 samples)
train - Saved 3111/4147
Skipped (too short): Ravdess_Augmented\4586.wav (194 samples)
train - Saved 3176/4147
Skipped (too short): Ravdess_Augmented\1955.wav (596 samples)
train - Saved 3837/4147
Skipped (too short): Ravdess_Augmented\4585.wav (155 samples)
train - Saved 3926/4147
Skipped (too short): Ravdess_Augmented\1954.wav (745 samples)
train - Saved 3931/4147
Skipped (too short): Ravdess_Augmented\4651.wav (221 samples)
train - Saved 4028/4147
Skipped (too short): Ravdess_Augmented\4649.wav (221 samples)
train - Saved 4136/4147
train Done — 4136 samples, 